# Preprocesamiento GPU (CUDA + OpenMP) en Google Colab
Entorno > Cambiar tipo de entorno de ejecucion > **GPU (T4)**. Luego ejecuta las celdas en orden.

In [ ]:
# 1) Verificar la GPU asignada por Colab
!nvidia-smi

## 2) Escribir el kernel CUDA + version CPU (OpenMP)

In [ ]:
%%writefile normalize.cu
#include <cstdio>
#include <vector>
#include <fstream>
#include <cuda_runtime.h>
#include <omp.h>
__global__ void normalizeKernel(float* d,int n,float mn,float range){int i=blockIdx.x*blockDim.x+threadIdx.x;if(i<n)d[i]=(d[i]-mn)/range;}
int main(int argc,char**argv){std::vector<float> h;{std::ifstream f(argv[1]);float v;while(f>>v)h.push_back(v);}int n=h.size();
 float mn=h[0],mx=h[0];
 #pragma omp parallel for reduction(min:mn) reduction(max:mx)
 for(int i=0;i<n;i++){if(h[i]<mn)mn=h[i];if(h[i]>mx)mx=h[i];}
 float range=(mx>mn)?(mx-mn):1.0f;float*d;cudaMalloc(&d,n*sizeof(float));cudaMemcpy(d,h.data(),n*sizeof(float),cudaMemcpyHostToDevice);
 cudaEvent_t a,b;cudaEventCreate(&a);cudaEventCreate(&b);int th=256,bl=(n+th-1)/th;cudaEventRecord(a);normalizeKernel<<<bl,th>>>(d,n,mn,range);cudaEventRecord(b);cudaEventSynchronize(b);
 float ms=0;cudaEventElapsedTime(&ms,a,b);cudaMemcpy(h.data(),d,n*sizeof(float),cudaMemcpyDeviceToHost);cudaFree(d);
 {std::ofstream f(argv[2]);for(int i=0;i<n;i++)f<<h[i]<<"\n";}printf("{\"device\":\"gpu\",\"n\":%d,\"ms\":%.4f}\n",n,ms);}


In [ ]:
%%writefile normalize_cpu.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>
int main(int c,char**v){FILE*fi=fopen(v[1],"r");int cap=1024,n=0;float*a=malloc(cap*4),x;while(fscanf(fi,"%f",&x)==1){if(n==cap){cap*=2;a=realloc(a,cap*4);}a[n++]=x;}fclose(fi);
 float mn=a[0],mx=a[0];
 #pragma omp parallel for reduction(min:mn) reduction(max:mx)
 for(int i=0;i<n;i++){if(a[i]<mn)mn=a[i];if(a[i]>mx)mx=a[i];}float r=(mx>mn)?(mx-mn):1.0f;
 double t0=omp_get_wtime();
 #pragma omp parallel for
 for(int i=0;i<n;i++)a[i]=(a[i]-mn)/r;double t1=omp_get_wtime();
 FILE*fo=fopen(v[2],"w");for(int i=0;i<n;i++)fprintf(fo,"%f\n",a[i]);fclose(fo);printf("{\"device\":\"cpu\",\"n\":%d,\"ms\":%.4f}\n",n,(t1-t0)*1000);}


## 3) Compilar y generar datos

In [ ]:
!nvcc -O2 -Xcompiler -fopenmp normalize.cu -o normalize
!gcc -O2 -fopenmp normalize_cpu.c -o normalize_cpu
import random
open('input.txt','w').write('\n'.join(str(random.randint(0,10000)) for _ in range(5_000_000)))
print('5M numeros generados')

## 4) Ejecutar y comparar GPU vs CPU (speedup)

In [ ]:
import subprocess,json
g=json.loads(subprocess.run(['./normalize','input.txt','out_gpu.txt'],capture_output=True,text=True).stdout)
c=json.loads(subprocess.run(['./normalize_cpu','input.txt','out_cpu.txt'],capture_output=True,text=True).stdout)
print('GPU:',g); print('CPU:',c)
print('Speedup GPU vs CPU =',round(c['ms']/g['ms'],2),'x')